# Model 05 — XGBoost

## BRFSS 2025 Diabetes Classification

**Purpose:** execute and freeze the fifth classifier without rerunning Models 01–04.

### Literature provenance

**BRFSS application 1:** Muhammad et al. (2025), *Journal of Primary Care & Community Health*, DOI `10.1177/21501319251400546`, included XGBoost among seven classifiers for self-reported diabetes on BRFSS 2023.

**BRFSS application 2:** Li, Peng, & Peng (2024), *PLOS ONE*, DOI `10.1371/journal.pone.0311222`, used XGBoost on BRFSS 2021 and later optimized it with a genetic algorithm.

**Architecture source:** Chen & Guestrin (2016), *XGBoost: A Scalable Tree Boosting System*, KDD '16, DOI `10.1145/2939672.2939785`.

This notebook evaluates a **plain fixed XGBoost baseline**, not GA-XGBoost and not a tuned reproduction of either BRFSS paper.


## 1. Frozen Study Protocol

Inherited unchanged from Part II:

- target: `DIABETE4=1` vs `DIABETE4=3`;
- 20 frozen predictors;
- 80/20 stratified split, `random_state=42`;
- categorical missing category + one-hot encoding;
- continuous median imputation + scaling;
- Stratified 5-Fold CV on training data only;
- natural class distribution;
- no SMOTE/undersampling;
- untouched test set used only after specification is frozen.

This allows a direct within-study comparison with Models 01–04.


In [ ]:
from pathlib import Path
from math import sqrt
import json
import platform
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import xgboost as xgb

from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "DIABETE4"

PRIMARY_FEATURES = [
    "_AGEG5YR", "_SEX", "_RACEGR3", "_EDUCAG", "_INCOMG1", "EMPLOY1",
    "_BMI5", "GENHLTH", "PHYSHLTH", "MENTHLTH", "_TOTINDA", "_RFSMOK3",
    "SLEPTIM1", "_RFHYPE6", "_RFCHOL3", "_MICHD", "CVDSTRK3",
    "CHCKDNY2", "DIFFWALK", "PERSDOC3",
]

CATEGORICAL_FEATURES = [
    "_AGEG5YR", "_SEX", "_RACEGR3", "_EDUCAG", "_INCOMG1", "EMPLOY1",
    "GENHLTH", "_TOTINDA", "_RFSMOK3", "_RFHYPE6", "_RFCHOL3", "_MICHD",
    "CVDSTRK3", "CHCKDNY2", "DIFFWALK", "PERSDOC3",
]

NUMERIC_FEATURES = ["_BMI5", "PHYSHLTH", "MENTHLTH", "SLEPTIM1"]

xpt_candidates = glob.glob("/kaggle/input/**/LLCP2025.XPT", recursive=True)
if len(xpt_candidates) != 1:
    raise RuntimeError(f"Expected exactly one LLCP2025.XPT input, found: {xpt_candidates}")
xpt_path = Path(xpt_candidates[0])

print("Input:", xpt_path)
print("Python:", platform.python_version())
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgb.__version__)


In [ ]:
df = pd.read_sas(xpt_path, format="xport", encoding="utf-8")

def keep_codes(series, valid_codes):
    return series.where(series.isin(valid_codes), np.nan)

def recode_brfss_2025(raw):
    out = pd.DataFrame(index=raw.index)

    out["_AGEG5YR"] = keep_codes(raw["_AGEG5YR"], range(1, 14))
    out["_SEX"] = keep_codes(raw["_SEX"], [1, 2])
    out["_RACEGR3"] = keep_codes(raw["_RACEGR3"], range(1, 6))
    out["_EDUCAG"] = keep_codes(raw["_EDUCAG"], range(1, 5))
    out["_INCOMG1"] = keep_codes(raw["_INCOMG1"], range(1, 8))
    out["EMPLOY1"] = keep_codes(raw["EMPLOY1"], range(1, 9))
    out["GENHLTH"] = keep_codes(raw["GENHLTH"], range(1, 6))
    out["_TOTINDA"] = keep_codes(raw["_TOTINDA"], [1, 2])
    out["_RFSMOK3"] = keep_codes(raw["_RFSMOK3"], [1, 2])
    out["_RFHYPE6"] = keep_codes(raw["_RFHYPE6"], [1, 2])
    out["_RFCHOL3"] = keep_codes(raw["_RFCHOL3"], [1, 2])
    out["_MICHD"] = keep_codes(raw["_MICHD"], [1, 2])
    out["CVDSTRK3"] = keep_codes(raw["CVDSTRK3"], [1, 2])
    out["CHCKDNY2"] = keep_codes(raw["CHCKDNY2"], [1, 2])
    out["DIFFWALK"] = keep_codes(raw["DIFFWALK"], [1, 2])
    out["PERSDOC3"] = keep_codes(raw["PERSDOC3"], [1, 2, 3])

    out["_BMI5"] = raw["_BMI5"] / 100.0

    for col in ["PHYSHLTH", "MENTHLTH"]:
        s = raw[col].replace({88.0: 0.0, 77.0: np.nan, 99.0: np.nan})
        out[col] = s.where(s.between(0, 30), np.nan)

    sleep = raw["SLEPTIM1"].replace({77.0: np.nan, 99.0: np.nan})
    out["SLEPTIM1"] = sleep.where(sleep.between(1, 24), np.nan)

    return out[PRIMARY_FEATURES]

target_mask = df[TARGET].isin([1.0, 3.0])
X = recode_brfss_2025(df.loc[target_mask, PRIMARY_FEATURES].copy())
y = (df.loc[target_mask, TARGET] == 1.0).astype("int8").rename("diabetes")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

assert len(X) == 342_539
assert len(X_train) == 274_031
assert len(X_test) == 68_508
assert int(y.sum()) == 51_827

print("Cohort:", len(X))
print("Train:", len(X_train), "Test:", len(X_test))
print("Train diabetes %:", y_train.mean() * 100)
print("Test diabetes %:", y_test.mean() * 100)


In [ ]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1.0)),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ],
    remainder="drop",
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

specificity_scorer = make_scorer(recall_score, pos_label=0)
precision_scorer = make_scorer(precision_score, zero_division=0)
scoring = {
    "accuracy": "accuracy",
    "precision": precision_scorer,
    "recall": "recall",
    "specificity": specificity_scorer,
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}


## 2. XGBoost Architecture

XGBoost extends gradient tree boosting with a regularized objective and systems optimizations for scalable tree construction. The original Chen & Guestrin formulation includes regularization on tree complexity and second-order optimization of the boosting objective.

The fixed baseline used here is:

```python
XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    booster="gbtree",
    n_estimators=100,
    learning_rate=0.3,
    max_depth=6,
    min_child_weight=1,
    gamma=0.0,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)
```

### Provenance classification

**Literature-supported**
- XGBoost model family and regularized boosting architecture;
- direct BRFSS diabetes use in P03 and P05;
- stratified CV / held-out evaluation.

**Study-specific fixed choices**
- 100 trees;
- learning rate 0.3;
- max depth 6;
- default-style L2 regularization;
- histogram tree builder for efficient execution;
- no SMOTE;
- threshold 0.5.

This is a plain XGBoost baseline, not GA-XGBoost and not a tuned reproduction.


In [ ]:
MODEL_05_DIR = Path("/kaggle/working/model_05_xgboost")
MODEL_05_DIR.mkdir(parents=True, exist_ok=True)

xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    booster="gbtree",
    n_estimators=100,
    learning_rate=0.3,
    max_depth=6,
    min_child_weight=1,
    gamma=0.0,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0,
)

xgb_pipeline = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", xgb_model),
])

config = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "xgboost": xgb.__version__,
    "model": "XGBClassifier",
    "architecture": "regularized gradient-boosted decision-tree ensemble",
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "booster": "gbtree",
    "n_estimators": 100,
    "learning_rate": 0.3,
    "max_depth": 6,
    "min_child_weight": 1,
    "gamma": 0.0,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "tree_method": "hist",
    "random_state": RANDOM_STATE,
    "threshold": 0.5,
    "resampling": "none",
    "cv_folds": 5,
}
with open(MODEL_05_DIR / "environment_and_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print(json.dumps(config, indent=2))


## 3. Five-Fold Cross-Validation

The complete preprocessing + XGBoost pipeline is re-fit independently inside each training fold.

No validation-fold or test-set information is used to select trees or preprocessing statistics.


In [ ]:
xgb_cv_raw = cross_validate(
    xgb_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1,
)

metric_order = ["accuracy", "precision", "recall", "specificity", "f1", "roc_auc", "pr_auc"]
t_critical_df4_95 = 2.7764451051977987

fold_rows = []
for fold in range(5):
    row = {"fold": fold + 1}
    for metric in metric_order:
        row[metric] = float(xgb_cv_raw[f"test_{metric}"][fold])
    row["fit_time_seconds"] = float(xgb_cv_raw["fit_time"][fold])
    row["score_time_seconds"] = float(xgb_cv_raw["score_time"][fold])
    fold_rows.append(row)

xgb_cv_folds = pd.DataFrame(fold_rows)

summary_rows = []
for metric in metric_order:
    values = xgb_cv_folds[metric].to_numpy()
    mean = float(values.mean())
    sd = float(values.std(ddof=1))
    margin = t_critical_df4_95 * (sd / sqrt(len(values)))
    summary_rows.append({
        "metric": metric,
        "mean": mean,
        "std": sd,
        "ci95_lower": max(0.0, mean - margin),
        "ci95_upper": min(1.0, mean + margin),
    })

xgb_cv_summary = pd.DataFrame(summary_rows)
display(xgb_cv_folds)
display(xgb_cv_summary)

xgb_cv_folds.to_csv(MODEL_05_DIR / "cv_folds.csv", index=False)
xgb_cv_summary.to_csv(MODEL_05_DIR / "cv_summary.csv", index=False)


## 4. Fit Frozen XGBoost and Evaluate the Untouched Test Set

After CV, the exact same fixed configuration is fit once on the full training set.

The test set is then evaluated once at threshold 0.5. No post-test parameter adjustment is permitted.


In [ ]:
xgb_pipeline.fit(X_train, y_train)

y_test_prob_xgb = xgb_pipeline.predict_proba(X_test)[:, 1]
y_test_pred_xgb = (y_test_prob_xgb >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred_xgb, labels=[0, 1]).ravel()

test_metrics = {
    "accuracy": float(accuracy_score(y_test, y_test_pred_xgb)),
    "precision": float(precision_score(y_test, y_test_pred_xgb, zero_division=0)),
    "recall": float(recall_score(y_test, y_test_pred_xgb)),
    "specificity": float(tn / (tn + fp)),
    "f1": float(f1_score(y_test, y_test_pred_xgb)),
    "roc_auc": float(roc_auc_score(y_test, y_test_prob_xgb)),
    "pr_auc": float(average_precision_score(y_test, y_test_prob_xgb)),
    "brier_score": float(brier_score_loss(y_test, y_test_prob_xgb)),
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    "threshold": 0.5,
}
print(json.dumps(test_metrics, indent=2))

with open(MODEL_05_DIR / "test_point_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2)


## 5. Stratified Bootstrap Confidence Intervals

Two hundred class-stratified bootstrap samples estimate uncertainty around the already-fixed held-out predictions. They are not used for tuning.


In [ ]:
BOOTSTRAP_REPS = 200
BOOTSTRAP_SEED = 20251007

rng = np.random.default_rng(BOOTSTRAP_SEED)
y_test_np = y_test.to_numpy()
pos_idx = np.flatnonzero(y_test_np == 1)
neg_idx = np.flatnonzero(y_test_np == 0)

bootstrap_rows = []
for b in range(BOOTSTRAP_REPS):
    sample_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)
    sample_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
    sample_idx = np.concatenate([sample_pos, sample_neg])
    rng.shuffle(sample_idx)

    y_b = y_test_np[sample_idx]
    p_b = y_test_prob_xgb[sample_idx]
    pred_b = (p_b >= 0.5).astype(int)

    tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_b, pred_b, labels=[0, 1]).ravel()
    bootstrap_rows.append({
        "replicate": b + 1,
        "accuracy": accuracy_score(y_b, pred_b),
        "precision": precision_score(y_b, pred_b, zero_division=0),
        "recall": recall_score(y_b, pred_b),
        "specificity": tn_b / (tn_b + fp_b),
        "f1": f1_score(y_b, pred_b),
        "roc_auc": roc_auc_score(y_b, p_b),
        "pr_auc": average_precision_score(y_b, p_b),
        "brier_score": brier_score_loss(y_b, p_b),
    })

bootstrap_df = pd.DataFrame(bootstrap_rows)
test_summary_rows = []
for metric in ["accuracy","precision","recall","specificity","f1","roc_auc","pr_auc","brier_score"]:
    values = bootstrap_df[metric].to_numpy()
    test_summary_rows.append({
        "metric": metric,
        "point_estimate": test_metrics[metric],
        "bootstrap_mean": float(values.mean()),
        "ci95_lower": float(np.quantile(values, 0.025)),
        "ci95_upper": float(np.quantile(values, 0.975)),
    })

test_summary = pd.DataFrame(test_summary_rows)
display(test_summary)

bootstrap_df.to_csv(MODEL_05_DIR / "bootstrap_metrics.csv", index=False)
test_summary.to_csv(MODEL_05_DIR / "test_summary.csv", index=False)


## 6. Diagnostics and Feature Importance

The ROC curve, precision-recall curve, confusion matrix, and transformed-feature importance are exported.

XGBoost feature importance is a model diagnostic only and is not causal evidence.


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_test_prob_xgb)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_test_prob_xgb)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"XGBoost (AUC={test_metrics['roc_auc']:.3f})")
plt.plot([0,1],[0,1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Model 05 — ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_05_DIR / "roc_curve.png", dpi=160)
plt.show()

plt.figure(figsize=(6,5))
plt.plot(recall_curve, precision_curve, label=f"XGBoost (AP={test_metrics['pr_auc']:.3f})")
plt.axhline(y=y_test.mean(), linestyle="--", label=f"Class prevalence={y_test.mean():.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Model 05 — Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_05_DIR / "precision_recall_curve.png", dpi=160)
plt.show()

cm = np.array([[tn, fp],[fn, tp]])
plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title("Model 05 — Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.xticks([0,1], ["No diabetes","Diabetes"])
plt.yticks([0,1], ["No diabetes","Diabetes"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm[i,j]:,}", ha="center", va="center")
plt.tight_layout()
plt.savefig(MODEL_05_DIR / "confusion_matrix.png", dpi=160)
plt.show()

feature_names = xgb_pipeline.named_steps["preprocess"].get_feature_names_out()
importances = xgb_pipeline.named_steps["model"].feature_importances_
importance_table = pd.DataFrame({
    "transformed_feature": feature_names,
    "xgboost_importance": importances,
}).sort_values("xgboost_importance", ascending=False)

display(importance_table.head(20))
importance_table.to_csv(MODEL_05_DIR / "feature_importance_transformed.csv", index=False)


## 7. Freeze Rule

After verified execution, Model 05 is frozen as:

> **100-tree XGBoost gbtree classifier, binary logistic objective, learning rate 0.3, max depth 6, L2 regularization 1.0, histogram tree builder, no resampling, threshold 0.5.**

No post-test change to depth, learning rate, regularization, sampling, threshold, preprocessing, or feature set is allowed.

Any GA-XGBoost, tuned XGBoost, class-weighted XGBoost, or resampled XGBoost must be recorded as a separate experiment.
